In [ ]:
import pandas as pd
import os
import shutil
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/it5006/data/Crimes_2015-2025.csv", header=0)

Mounted at /content/drive


In [ ]:
# drop rows with missing value
missing = data[['Date', 'Latitude', 'Longitude']].isna().any(axis=1).sum()
print("Missing values:", missing)

data.dropna(subset=['Date', 'Latitude', 'Longitude'], inplace=True)

Missing values: 42337


In [ ]:
# Convert time in the Date column to two new columns: Year, Month
data['Date'] = pd.to_datetime(data['Date'])
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['DayOfWeek'] = data['Date'].dt.day_name()
data['Hour'] = data['Date'].dt.hour

data.head()

/tmp/ipython-input-4155349992.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Date'] = pd.to_datetime(data['Date'])


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location,Month,DayOfWeek,Hour
0,14075483,JK105557,2025-12-31 23:58:00,050XX S PAULINA ST,0560,ASSAULT,SIMPLE,RESIDENCE,False,False,...,1165860.0,1871343.0,2025,01/09/2026 03:40:47 PM,41.802549,-87.667246,"(41.802549018, -87.667246428)",12,Wednesday,23
1,14070833,JK100050,2025-12-31 23:55:00,053XX W WASHINGTON BLVD,0930,MOTOR VEHICLE THEFT,THEFT / RECOVERY - AUTOMOBILE,APARTMENT,False,True,...,1140809.0,1900235.0,2025,01/08/2026 03:46:48 PM,41.882329,-87.758411,"(41.882328854, -87.758411303)",12,Wednesday,23
2,14070745,JK100011,2025-12-31 23:54:00,100XX W OHARE ST,2890,PUBLIC PEACE VIOLATION,OTHER VIOLATION,AIRCRAFT,False,False,...,1100658.0,1934241.0,2025,01/08/2026 03:46:48 PM,41.976290,-87.905227,"(41.976290414, -87.905227221)",12,Wednesday,23
3,14070799,JK100014,2025-12-31 23:54:00,100XX W OHARE ST,2890,PUBLIC PEACE VIOLATION,OTHER VIOLATION,AIRCRAFT,False,False,...,1100658.0,1934241.0,2025,01/08/2026 03:46:48 PM,41.976290,-87.905227,"(41.976290414, -87.905227221)",12,Wednesday,23
4,14070845,JK100006,2025-12-31 23:54:00,013XX W LAKE ST,0454,BATTERY,"AGGRAVATED P.O. - HANDS, FISTS, FEET, NO / MIN...",RESTAURANT,True,False,...,1167120.0,1901555.0,2025,01/08/2026 03:46:48 PM,41.885427,-87.661759,"(41.885426714, -87.661759042)",12,Wednesday,23


In [ ]:
OUTPUT_DIR = 'data_chunks'
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

In [ ]:
for col in ['Primary Type', 'Description', 'Location Description', 'DayOfWeek']:
    if col in data.columns:
        data[col] = data[col].astype('category')

data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2713480 entries, 0 to 2755816
Data columns (total 25 columns):
 #   Column                Dtype         
---  ------                -----         
 0   ID                    int64         
 1   Case Number           object        
 2   Date                  datetime64[ns]
 3   Block                 object        
 4   IUCR                  object        
 5   Primary Type          category      
 6   Description           category      
 7   Location Description  category      
 8   Arrest                bool          
 9   Domestic              bool          
 10  Beat                  int64         
 11  District              float64       
 12  Ward                  float64       
 13  Community Area        float64       
 14  FBI Code              object        
 15  X Coordinate          float64       
 16  Y Coordinate          float64       
 17  Year                  int32         
 18  Updated On            object        
 19  Latit

In [ ]:
unique_years = sorted(data['Year'].unique())

for year in unique_years:
    year_data = data[data['Year'] == year].copy()
    save_path = os.path.join(OUTPUT_DIR, f"crimes_{year}.parquet")

    year_data.to_parquet(save_path, engine='pyarrow', compression='snappy')

    file_size = os.path.getsize(save_path) / (1024 * 1024)
    print(f"   - ✅ {year} data saved ({len(year_data):>6,} rows) |  {file_size:.2f} MB")

   - ✅ 2015 data saved (257,908 rows) |  16.83 MB
   - ✅ 2016 data saved (267,224 rows) |  17.32 MB
   - ✅ 2017 data saved (264,936 rows) |  17.07 MB
   - ✅ 2018 data saved (263,523 rows) |  17.10 MB
   - ✅ 2019 data saved (259,170 rows) |  16.88 MB
   - ✅ 2020 data saved (207,970 rows) |  14.07 MB
   - ✅ 2021 data saved (202,869 rows) |  13.64 MB
   - ✅ 2022 data saved (234,884 rows) |  15.36 MB
   - ✅ 2023 data saved (261,243 rows) |  16.95 MB
   - ✅ 2024 data saved (257,553 rows) |  16.63 MB
   - ✅ 2025 data saved (236,200 rows) |  15.45 MB


In [ ]:
source_folder = '/content/data_chunks'
destination_folder = '/content/drive/MyDrive/Colab Notebooks/it5006/data_chunks'

try:
    if os.path.exists(destination_folder):
        shutil.rmtree(destination_folder)

    shutil.copytree(source_folder, destination_folder)
except Exception as e:
    print(f"❌ failed: {e}")

In [11]:
# Data Verification
import glob

files = sorted(glob.glob(os.path.join(destination_folder, "crimes_*.parquet")))

total_rows = 0
years_found = []

for f in files:
    try:
        filename = os.path.basename(f)
        year = int(filename.replace('crimes_', '').replace('.parquet', ''))
        years_found.append(year)

        df = pd.read_parquet(f)
        count = len(df)
        total_rows += count

        size_mb = os.path.getsize(f) / (1024 * 1024)

        print(f"{filename:<30} | {year:<6} | {count:<10,} | {size_mb:.2f}")

    except Exception as e:
        print(f"❌ read file {f} failed: {e}")

print("-" * 70)
print(f"📊 total rows: {total_rows:,}")
print(f"📅 slot: {min(years_found)} - {max(years_found)}")

crimes_2015.parquet            | 2015   | 257,908    | 16.83
crimes_2016.parquet            | 2016   | 267,224    | 17.32
crimes_2017.parquet            | 2017   | 264,936    | 17.07
crimes_2018.parquet            | 2018   | 263,523    | 17.10
crimes_2019.parquet            | 2019   | 259,170    | 16.88
crimes_2020.parquet            | 2020   | 207,970    | 14.07
crimes_2021.parquet            | 2021   | 202,869    | 13.64
crimes_2022.parquet            | 2022   | 234,884    | 15.36
crimes_2023.parquet            | 2023   | 261,243    | 16.95
crimes_2024.parquet            | 2024   | 257,553    | 16.63
crimes_2025.parquet            | 2025   | 236,200    | 15.45
----------------------------------------------------------------------
📊 total rows: 2,713,480
📅 slot: 2015 - 2025
